# Notebook 04: Baseline Models
This notebook trains and compares 6 traditional machine learning baselines: RF, XGBoost, LightGBM, SVM, Logistic Regression, and MLP.

In [1]:
import os
import pickle
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

print("Baseline packages loaded!")


Baseline packages loaded!


In [2]:
# Load fused features
X_flat = np.load('datasets/processed/X_flat.npy')
y = np.load('datasets/processed/y.npy')

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X_flat, y, test_size=0.2, random_state=42)

# Scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

os.makedirs('models', exist_ok=True)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"Data loaded. Train shape: {X_train.shape}, Val shape: {X_val.shape}")


Data loaded. Train shape: (80, 11300), Val shape: (20, 11300)


In [3]:
models = {
    'random_forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'xgboost': XGBClassifier(n_estimators=50, max_depth=4, random_state=42, eval_metric='logloss'),
    'lightgbm': LGBMClassifier(n_estimators=50, max_depth=4, random_state=42, verbose=-1),
    'svm': SVC(probability=True, random_state=42),
    'logistic_regression': LogisticRegression(max_iter=500, random_state=42),
    'mlp': MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, random_state=42)
}

metrics = {}
os.makedirs('reports', exist_ok=True)

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds)
    prec = precision_score(y_val, preds)
    rec = recall_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    
    metrics[name] = {
        'accuracy': float(acc),
        'f1': float(f1),
        'precision': float(prec),
        'recall': float(rec),
        'auc': float(auc)
    }
    
    # Save model pickle
    with open(f'models/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
        
df_metrics = pd.DataFrame(metrics).T
df_metrics.to_json('reports/baseline_metrics.json')
print("Finished baseline model training! Leaderboard:")
print(df_metrics)


Training random_forest...
Training xgboost...


Training lightgbm...


C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training svm...


Training logistic_regression...


Training mlp...


Finished baseline model training! Leaderboard:
                     accuracy        f1  precision  recall       auc
random_forest             0.9  0.947368        0.9     1.0  0.236111
xgboost                   0.9  0.947368        0.9     1.0  0.361111
lightgbm                  0.9  0.947368        0.9     1.0  0.583333
svm                       0.9  0.947368        0.9     1.0  0.750000
logistic_regression       0.9  0.947368        0.9     1.0  0.416667
mlp                       0.9  0.947368        0.9     1.0  0.250000
